In [1]:
%load_ext autoreload
%autoreload 2
%cd /home/albin/egna_proj/block_puzzle_rl/

/home/albin/egna_proj/block_puzzle_rl


/home/albin/egna_proj/block_puzzle_rl/.venv/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [2]:
import numpy as np
import torch
import gymnasium as gym
from stable_baselines3 import DQN
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.vec_env import DummyVecEnv, SubprocVecEnv
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.monitor import Monitor
from gymnasium import spaces
from stable_baselines3.common.callbacks import BaseCallback
import numpy as np
from collections import deque
from stable_baselines3.common.callbacks import CallbackList
from sb3_contrib import MaskablePPO
from game.plot_game import render_text
from sb3_contrib.common.maskable.utils import get_action_masks
from sb3_contrib.common.wrappers import ActionMasker
import os
from datetime import datetime
from game.block_puzzle_env import BlockPuzzleEnv
from agent.utils import encode_state, encode_state_cnn_modern
from sb3_utils.callbacks import AveragedMetricsCallback, SaveEveryNTimestepsCallback, UnfreezeCallback
import torch
import torch.nn as nn
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from game.plot_game import render_text_with_blocks

In [3]:
class DiscreteActionWrapper(gym.Env):
    def __init__(self, raw_env):
        super().__init__()
        self.raw_env = raw_env
        self.original_action_space = raw_env.action_space  # Should be MultiDiscrete
        self.obs_space = self.raw_env.observation_space
        
        dummy_obs, _ = self.raw_env.reset()
        grid, blocks = encode_state_cnn_modern(dummy_obs)


        # Flatten MultiDiscrete([a, b, c]) → Discrete(a * b * c)
        self.observation_space = spaces.Dict({
            'grid': spaces.Box(low=0.0, high=1.0, shape=(grid.shape), dtype=np.float32),
            'blocks': spaces.Box(low=0.0, high=1.0, shape=(blocks.shape), dtype=np.float32),
        })

        self.action_space = spaces.Discrete(np.prod(self.original_action_space.nvec))

    def reset(self, **kwargs):
        obs_dict, _ = self.raw_env.reset(**kwargs)
        grid, blocks = encode_state_cnn_modern(obs_dict)
        return {'grid': grid.astype(np.float32), 'blocks': blocks.astype(np.float32)}, {}

    def step(self, flat_action):
        a0, a1, a2 = np.unravel_index(flat_action, self.original_action_space.nvec)
        action = (int(a0), int(a1), int(a2))
        (obs_dict, _), reward, terminated, truncated, info = self.raw_env.step(action)
        grid, blocks = encode_state_cnn_modern(obs_dict)
        return {'grid': grid.astype(np.float32), 'blocks': blocks.astype(np.float32)}, reward, terminated, truncated, info

    def render(self, **kwargs):
        return self.raw_env.render(**kwargs)



In [4]:
raw_env = BlockPuzzleEnv(width=8, height=10, num_blocks=3)
wrapped_env = DiscreteActionWrapper(raw_env)
def mask_fn(env):
    return env.raw_env.game.compute_action_mask()
masked_env = ActionMasker(wrapped_env, mask_fn)
check_env(masked_env, warn=True) 

/home/albin/egna_proj/block_puzzle_rl/.venv/lib/python3.10/site-packages/stable_baselines3/common/env_checker.py:272: UserWarning: Your observation blocks has an unconventional shape (neither an image, nor a 1D vector). We recommend you to flatten the observation to have only a 1D vector or use a custom policy to properly process the data.
  warnings.warn(
/home/albin/egna_proj/block_puzzle_rl/.venv/lib/python3.10/site-packages/stable_baselines3/common/env_checker.py:272: UserWarning: Your observation grid has an unconventional shape (neither an image, nor a 1D vector). We recommend you to flatten the observation to have only a 1D vector or use a custom policy to properly process the data.
  warnings.warn(


In [5]:
class BlockPointerFeatures(BaseFeaturesExtractor):
    """
    Feature extractor + action-logic for Blodoku:
      - CNN on grid
      - Shared MLP on each block one-hot → embedding
      - Pointer-attention to choose which block
      - Sub-action head to choose placement position
      - Outputs flat logits over (block, position) pairs
    """
    def __init__(
        self,
        observation_space: gym.spaces.Dict,
        cnn_channels=[16, 32, 32, 64, 64],
        block_embedding_dim=64,
    ):  
        # Calculate total actions: num_blocks * grid_cells
        H, W = observation_space.spaces['grid'].shape
        self.num_blocks = observation_space.spaces['blocks'].shape[0]
        self.num_positions = H * W

        # temporary features_dim=1, will override
        super().__init__(observation_space, features_dim=1)

        # 1) CNN for grid
        in_ch = 1
        cnn_modules = []
        for out_ch in cnn_channels:
            cnn_modules += [nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1), nn.ReLU()]
            in_ch = out_ch
        cnn_modules.append(nn.Flatten())
        self.cnn = nn.Sequential(*cnn_modules)

        # Compute CNN output dimension
        with torch.no_grad():
            dummy = torch.zeros(1, 1, H, W)
            cnn_out_dim = self.cnn(dummy).shape[1]

        # 2) Block embedding MLP
        P = observation_space.spaces['blocks'].shape[-1]
        self.block_mlp = nn.Sequential(
            nn.Linear(P, 128), nn.ReLU(),
            nn.Linear(128, block_embedding_dim), nn.ReLU(),
        )

        # 3) Query network: grid features → query vector
        self.query_net = nn.Sequential(
            nn.Linear(cnn_out_dim, block_embedding_dim), nn.ReLU(),
            nn.Linear(block_embedding_dim, block_embedding_dim),
        )

        # 4) Sub-action head: embedding → position logits
        self.subaction_head = nn.Linear(block_embedding_dim, self.num_positions)

        # Final output dim: num_blocks * num_positions
        self._features_dim = self.num_blocks * self.num_positions

    def forward(self, observations: dict) -> torch.Tensor:
        # Grid through CNN
        grid = observations['grid'].unsqueeze(1)  # (B,1,H,W)
        feat = self.cnn(grid)                     # (B, F)

        # Blocks embedding
        blocks = observations['blocks']          # (B, N, P)
        B, N, P = blocks.shape
        flat = blocks.view(B * N, P)             # (B*N, P)
        e_flat = self.block_mlp(flat)            # (B*N, D)
        e = e_flat.view(B, N, -1)                # (B, N, D)

        # Pointer logits
        q = self.query_net(feat)                 # (B, D)
        slot_logits = torch.einsum('bd,bnd->bn', q, e)  # (B, N)

        # Sub-action logits
        sub_flat = self.subaction_head(e_flat)   # (B*N, H*W)
        sub = sub_flat.view(B, N, self.num_positions)  # (B, N, H*W)

        # Combine and flatten
        joint = slot_logits.unsqueeze(-1) + sub  # (B, N, H*W)
        flat_logits = joint.view(B, -1)          # (B, N*H*W)
        return flat_logits


In [6]:
def lr_warmup_schedule(progress_remaining) -> float:
    """
    SB3 passes in `progress_remaining` which goes from 1.0 → 0.0 over the entire learn() call.
    We want:
      - first 10% (progress 1.0 → 0.9): lr goes 1e-5 → 1e-3
      - remaining 90% (progress 0.9 → 0.0): lr goes 1e-3 → 1e-4
    """
    max_lr = 1e-3
    init_lr = 1e-5
    final_lr = 1e-4
    warmup_percent = 0.1  

    # Calculate how far we are into training [0.0 .. 1.0]
    t = 1.0 - progress_remaining

    if t < warmup_percent:
        # warm-up phase: 
        return init_lr + (max_lr - init_lr) * (t / warmup_percent)
    else:
        # decay phase: 
        decay_t = (t - warmup_percent) / (1.0 - warmup_percent)  
        return max_lr + (final_lr - max_lr) * decay_t

In [7]:
def make_env():
    def _init():
        raw_env = BlockPuzzleEnv(width=8, height=10, num_blocks=3)
        wrapped_env = DiscreteActionWrapper(raw_env)

        def mask_fn(env):
            return env.raw_env.game.compute_action_mask()

        masked_env = ActionMasker(wrapped_env, mask_fn)
        monitored_env = Monitor(masked_env)
        return monitored_env

    return _init

n_envs = 48
vec_env = SubprocVecEnv([make_env() for _ in range(n_envs)])

policy_kwargs = dict(
    features_extractor_class  = BlockPointerFeatures,
    features_extractor_kwargs = dict(
        cnn_channels        = [16, 32, 32, 64, 64],   # or whatever you prefer
        block_embedding_dim = 64,
    ),
)

model = MaskablePPO(
    policy            = "MultiInputPolicy",
    env               = vec_env,
    learning_rate     = lr_warmup_schedule,
    n_steps           = 256,
    batch_size        = 4096,
    gamma             = 0.90,
    device            = "cuda",
    verbose           = 1,
    tensorboard_log   = "./sb3_logs/",
    policy_kwargs     = policy_kwargs,
)

callback_list = CallbackList([
    AveragedMetricsCallback(), 
    SaveEveryNTimestepsCallback(save_freq=250_000, save_path="./crash_backup_save/", name="backup_save", with_time=False, with_timesteps=False),
])

load_weights = False
weight_path = ""
if load_weights:
    model.set_parameters(weight_path, exact_match=False)


model.learn(total_timesteps=100_000_000, callback=callback_list, tb_log_name="PPO_CNN_MODERN_MASKED_QUERY")
model.save("sb3_block_ppo_cnn_modern_masked_query")

Using cuda device
Logging to ./sb3_logs/PPO_CNN_MODERN_MASKED_QUERY_1
-----------------------------------
| custom/              |          |
|    cleared_lines_avg | 1.03     |
|    invalid_moves_avg | 0        |
|    move_count_avg    | 17.3     |
| rollout/             |          |
|    ep_len_mean       | 17.3     |
|    ep_rew_mean       | -8.45    |
| time/                |          |
|    fps               | 3876     |
|    iterations        | 1        |
|    time_elapsed      | 3        |
|    total_timesteps   | 12288    |
-----------------------------------
------------------------------------------
| custom/                 |              |
|    cleared_lines_avg    | 1.27         |
|    invalid_moves_avg    | 0            |
|    move_count_avg       | 17.5         |
| rollout/                |              |
|    ep_len_mean          | 17.5         |
|    ep_rew_mean          | -8.02        |
| time/                   |              |
|    fps                  | 3864       

KeyboardInterrupt: 

In [ ]:
model.save("sb3_block_ppo_cnn_modern_masked_query")

In [ ]:
# callback_list = CallbackList([
#     AveragedMetricsCallback(), 
#     SaveEveryNTimestepsCallback(save_freq=1_000_000, save_path="./sb3_model_saves/", name="sb3_block_ppo_mlp_masked_rewards2"),
#     SaveEveryNTimestepsCallback(save_freq=100_000, save_path="./crash_backup_save/", name="backup_save", with_time=False, with_timesteps=False),
# ])

# for i in range(0, 50):
#     if i != 0:
#         backup_path = "crash_backup_save/backup_save__"
#         model.set_parameters(backup_path, exact_match=True)

#     try:
#         model.learn(total_timesteps=30_000_001, callback=callback_list, tb_log_name="PPO_MLP_MASKED")
#     except Exception as e:
#         pass

In [ ]:

single_env = DiscreteActionWrapper(BlockPuzzleEnv(width=8, height=10, num_blocks=3))
obs, _ = single_env.reset()
done = False
total_reward = 0
step = 0

while not done:
    # Get valid action mask
    mask = single_env.raw_env.game.compute_action_mask()
        
    # Use model.predict with action_masks for MaskablePPO
    action, _ = model.predict(obs, action_masks=mask, deterministic=True)
    
    obs, reward, terminated, truncated, info = single_env.step(action)
    total_reward += reward
    done = terminated or truncated
    step += 1
    
    game_grid = single_env.raw_env.game.grid.get_game_grid()
    game_blocks = [block.grid() for block in single_env.raw_env.game.block_queue]

    render_text_with_blocks(game_grid, game_blocks)
    print()


print(f"Evaluation finished in {step} steps, total reward: {total_reward}")


In [ ]:
game_grid